GUI Preprocessing: Mask
=======================

This tool allows one to mask a bespoke mask for a given image of a strong lens using an interactive GUI. This mask
can then be loaded before a pipeline is run and passed to that pipeline so as to become the default masked used by a
search (if a mask function is not passed to that search).

This GUI is adapted from the following code: https://gist.github.com/brikeats/4f63f867fd8ea0f196c78e9b835150ab

__Contents__

- **Dataset:** Load and plot the strong lens dataset.
- **Scribbler:** Load the Scribbler GUI for drawing the mask.
- **Refining An Existing Mask:** Reopen a saved mask as a proposal and add to / erase from it.
- **Output:** Now lets plot the image and mask, so we can check that the mask includes the regions of the image.

__Google Colab Setup__

This cell sets up the environment when the notebook is run on Google Colab: it installs the
required PyAuto packages, clones the workspace (configuration files and example datasets) and
points the configuration at it. If you are running the notebook elsewhere (e.g. locally via
your own installation) it does nothing, and you can run it safely.

Colab tip: model-fits run much faster on a GPU — enable one via "Runtime" -> "Change runtime
type" -> "Hardware accelerator" before running the notebook.

In [ ]:
try:
    import google.colab
except ImportError:
    from autolens import setup_colab as _setup_colab
else:
    import importlib
    import subprocess
    import sys

    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "autonerves", "--no-deps"]
    )
    _setup_colab = importlib.import_module("autonerves.setup_colab")

_setup_colab.setup("autolens")

In [ ]:

from autolens import jax_wrapper  # Sets JAX environment before other imports

from autolens import setup_notebook; setup_notebook()

from pathlib import Path
import autolens as al
import autolens.plot as aplt
import numpy as np

__Dataset__

Setup the path the datasets we'll use to illustrate preprocessing, which is the 
folder `dataset/imaging/simple__no_lens_light`.

In [ ]:
dataset_name = "simple__no_lens_light"
dataset_path = Path("dataset") / "imaging" / dataset_name

__Dataset Auto-Simulation__

If the dataset does not already exist on your system, it will be created by running the corresponding
simulator script. This ensures that all example scripts can be run without manually simulating data first.

In [ ]:
if al.util.dataset.should_simulate(str(dataset_path)):
    import subprocess
    import sys

    subprocess.run(
        [sys.executable, "scripts/imaging/features/no_lens_light/simulator.py"],
        check=True,
    )

The pixel scale of the imaging dataset.

In [ ]:
pixel_scales = 0.1

Load the `Imaging` dataset, so that the mask can be plotted over the strong lens image.

In [ ]:
data = al.Array2D.from_fits(
    file_path=dataset_path / "data.fits", pixel_scales=pixel_scales
)

__Scribbler__

Load the Scribbler GUI for drawing the mask, painting over the region of the image you want to fit.

Two brushes are available: press `1` for the white brush, which ADDS pixels to the painted region, and `2` for the black
brush, which ERASES them. Press `=` / `-` to make the brush bigger / smaller (each press scales it by 1.4x), `z` to
undo the last stroke and Esc when you are finished.

`mask_from()` returns everything painted white that was not painted black.

If faint structure is hidden under the galaxy's light, pass `subtract_radial=True` to also show the image with its
azimuthally-averaged radial profile subtracted, side by side with the image as observed. This changes only the
display, not the mask, and you can paint on either panel.

If the multiple-image positions of the lensed source have already been marked with the `positions.py` GUI, pass
them as `positions` and each is marked with a dark cross while you paint, so the mask is drawn around the same
images. The crosses are display only and never enter the mask.

__Refining An Existing Mask__

To adjust a mask drawn previously instead of starting from a blank image, set `refine_existing = True`. If the
saved mask exists it is loaded and passed to the GUI as a `proposal`, instead of opening a blank canvas. Its
boundary is outlined in black over the image, and `mask_from()` then returns the proposal plus whatever you paint
white, minus whatever you paint black. With `refine_existing = False` (or no saved mask yet) the GUI opens on the
blank image.

The `.fits` written at the end of this script stores the region to *exclude* (`True` = masked), so it is inverted
back to the painted region before being passed as the proposal, and the result is inverted again afterwards.

In [ ]:
refine_existing = False

mask_path = Path(dataset_path, "mask_gui.fits")
positions_path = Path(dataset_path, "positions.json")

positions = al.from_json(file_path=positions_path) if positions_path.exists() else None

if refine_existing and mask_path.exists():
    previous = al.Mask2D.from_fits(file_path=mask_path, pixel_scales=pixel_scales)
    scribbler = al.Scribbler(
        image=data.native,
        proposal=np.invert(np.asarray(previous)),
        positions=positions,
        title="Mask: paint the region to FIT, refining the saved mask",
    )
else:
    scribbler = al.Scribbler(
        image=data.native,
        positions=positions,
        title="Mask: paint the region to FIT",
    )

painted = scribbler.mask_from()

If the window was closed without painting anything (or everything painted was erased again) there is no region to
fit: inverted, it would mask every pixel. Stop here rather than write such a mask.

In [ ]:
if not painted.any():
    raise SystemExit("No region was painted, so no mask is written.")

mask = al.Mask2D(mask=np.invert(painted), pixel_scales=pixel_scales)

__Output__

Now lets plot the image and mask, so we can check that the mask includes the regions of the image we want.

In [ ]:
aplt.plot_array(array=data, title="")

Output this image of the mask to a .png file in the dataset folder for future reference.

In [ ]:
aplt.plot_array(array=data, title="")

Output it to the dataset folder of the lens, so that we can load it from a .fits in our modeling scripts.

In [ ]:
aplt.fits_array(
    array=mask, file_path=Path(dataset_path, "mask_gui.fits"), overwrite=True
)
